In [61]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import contextlib
import os
import io
from data_profiling import ProfileReport

In [5]:
with open("../hw1_train.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

In [6]:
len(data)

10000

In [10]:
sample = data[0]

print("ID:", sample["dialog_id"])
print("Language:", sample["language"])
print("User turns:", sample["n_user_turns"])
print("Reaction:", sample["labels"]["reaction"])
print("Task type:", sample["labels"]["task_type"])

ID: d094012177e
Language: en
User turns: 7
Reaction: positive
Task type: analysis


In [33]:
labeled = [
    d for d in data
    if isinstance(d.get("labels"), dict)
    and d["labels"].get("reaction") is not None
]

unlabeled = [
    d for d in data
    if not (
        isinstance(d.get("labels"), dict)
        and d["labels"].get("reaction") is not None
    )
]

print("Всего диалогов:", len(data))
print("С разметкой reaction:", len(labeled))
print("Без разметки reaction:", len(unlabeled))

Всего диалогов: 10000
С разметкой reaction: 3000
Без разметки reaction: 7000


In [ ]:
none_labels = [
    d for d in data
    if d.get("labels") is None
]

print("Количество labels=None:", len(none_labels))

for d in none_labels[:3]:
    print("\nID:", d.get("dialog_id"))
    print("Язык:", d.get("language"))
    print("Количество user turns:", d.get("n_user_turns"))
    print("\n--- Сообщения ---")
    for turn in d.get("turns", []):
        role = turn.get("role", "unknown")
        text = turn.get("text", "")
        print(f"[{role}]: {text}")

Количество labels=None: 7000

ID: d4453789139
Язык: en
Количество user turns: 8

--- Сообщения ---
[user]: prog like picasa but more powerfull
[assistant]: There are several powerful alternatives to Picasa that offer advanced photo editing and management features. Some popular options include:

1. Adobe Lightroom: Lightroom is a professional-grade photo editing and management software developed by Adobe. It offers a wide range of editing tools, advanced organization features, and integration with Adobe Creative Cloud.

2. Capture One: Capture One is another powerful photo editing and management software designed for professional photographers. It provides advanced editing tools, raw image processing, and efficient organizational features.

3. DxO PhotoLab: DxO PhotoLab is known for its powerful editing capabilities and high-quality image processing algorithms. It offers advanced features like lens correction and noise reduction, while also providing a user-friendly interface.

4. ACDSe

In [34]:
label_dicts = [
    d["labels"]
    for d in data
    if isinstance(d.get("labels"), dict)
]

for key in ["task_type", "reaction", "user_turns"]:
    count = sum(key in labels for labels in label_dicts)
    print(f"{key}: {count}")

task_type: 3000
reaction: 3000
user_turns: 3000


In [35]:
reactions = pd.Series([
    d["labels"]["reaction"]
    for d in labeled
])

print("Количество размеченных диалогов:", len(reactions))

print("\nКоличество объектов каждого класса:")
print(reactions.value_counts())

print("\nДоли классов:")
print(reactions.value_counts(normalize=True))

Количество размеченных диалогов: 3000

Количество объектов каждого класса:
neutral     1774
negative     923
positive     303
Name: count, dtype: int64

Доли классов:
neutral     0.591333
negative    0.307667
positive    0.101000
Name: proportion, dtype: float64


In [37]:
allowed = {"positive", "neutral", "negative"}

unexpected = set(reactions.dropna()) - allowed

print("Неожиданные значения reaction:", unexpected)

Неожиданные значения reaction: set()


In [56]:
labeled_dialogs = pd.DataFrame(labeled)

labels_expanded = pd.json_normalize(labeled_dialogs['labels'])

labeled_dialogs = pd.concat(
    [labeled_dialogs.drop(columns=['labels']), labels_expanded], 
    axis=1
)
labeled_dialogs

,dialog_id,language,n_user_turns,turns,task_type,reaction,user_turns
0,d094012177e,en,7,"[{'role': 'user', 'text': 'it’s like: 1+1=2, b...",analysis,positive,"[{'task_type': 'analysis', 'state': 'continuat..."
1,d306fbe61fe,en,8,"[{'role': 'user', 'text': 'Can you help me tra...",information_seeking,neutral,"[{'task_type': 'information_seeking', 'state':..."
2,dd6905952c7,en,5,"[{'role': 'user', 'text': 'ask questions about...",creation,neutral,"[{'task_type': 'creation', 'state': 'newtopic'..."
3,d767a675896,en,6,"[{'role': 'user', 'text': 'Calculate each prom...",information_seeking,negative,"[{'task_type': 'information_seeking', 'state':..."
4,d7a6280ce8e,en,8,"[{'role': 'user', 'text': 'mass spectrometer, ...",information_seeking,negative,"[{'task_type': 'information_seeking', 'state':..."
...,...,...,...,...,...,...,...
2995,ddf17ced020,ru,8,"[{'role': 'user', 'text': 'И здесь К-ый максим...",information_seeking,negative,"[{'task_type': 'open-ended_discovery', 'state'..."
2996,d1f0a4cb059,ru,5,"[{'role': 'user', 'text': 'What languages do y...",creation,neutral,"[{'task_type': 'information_seeking', 'state':..."
2997,dc047dadf97,ru,5,"[{'role': 'user', 'text': 'Фрукты Пользователь...",creation,neutral,"[{'task_type': 'creation', 'state': 'newtopic'..."
2998,da0ba798a15,ru,5,"[{'role': 'user', 'text': 'напиши скрипт на py...",information_seeking,negative,"[{'task_type': 'creation', 'state': 'newtopic'..."


In [ ]:

profile = ProfileReport(labeled_dialogs, title="Описание размеченых диалогов", progress_bar=False)
profile_path = os.path.join("", "labeled_dialogs.html")
with contextlib.redirect_stderr(io.StringIO()):
    profile.to_file(profile_path)
print("отчет:", profile_path)

In [ ]:
profile = ProfileReport(labeled_dialogs, title="Описание размеченых диалогов", progress_bar=False)
profile_path = os.path.join("", "labeled_dialogs.html")
with contextlib.redirect_stderr(io.StringIO()):
    profile.to_file(profile_path)
print("отчет:", profile_path)

отчет: labeled_dialogs.html
